In [1]:
from jqdatasdk import *
import configparser
 
config = configparser.ConfigParser()
file = 'config.ini'
config.read(file)
user = config.get('login','user')
password = config.get('login','password')
auth(user,password)

auth success （JQData现有流量增加活动，详情请咨询JQData管理员，微信号：JQData02）


## 获取基金数据

我们目前包含多种ETF、LOF、分级A/B基金完整的行情、净值数据. 详情见[基金列表](https://www.joinquant.com/help/api/help?name=fund#%E5%9F%BA%E9%87%91%E5%88%97%E8%A1%A8)

更多API的可参考[官方API文档](https://www.joinquant.com/api)

### 基金概况：上市退市时间、代码、名称、累计/单位净值等

#### 获取单支基金数据

获取单支基金的信息.

**调用方法**

```
get_security_info(code)
```

**参数**

- code: 基金代码

**返回值**

\- 一个对象, 有如下属性:

- display_name # 中文名称
- name # 缩写简称
- start_date # 上市日期, [datetime.date] 类型
- end_date # 退市日期， [datetime.date] 类型, 如果没有退市则为2200-01-01
- type # 类型 etf(ETF基金)，fja（分级A），fjb（分级B）,fjm(分级母基金），lof(场内交易开发基金），mmf(场内交易货币基金）
- parent # 分级基金的母基金代码

**示例**

In [ ]:
get_security_info('159901.XSHE').start_date #获取深100ETF的上市日期

### 获取所有基金数据
获取平台支持的所有基金数据

**调用方法**

get_all_securities(['fund'])  
这里请在使用时注意防止未来函数。

**返回**
- display_name # 中文名称
- name # 缩写简称
- start_date # 上市日期
- end_date # 退市日期，如果没有退市则为2200-01-01
- type # 类型， etf(ETF基金)，fja（分级A），fjb（分级B）,fjm(分级母基金），lof(场内交易开发基金），mmf(场内交易货币基金） [pandas.DataFrame], 比如:get_all_securities(['fund'])[:2]返回:

In [7]:
get_all_securities(['fund'])[:2]

,display_name,name,start_date,end_date,type
150008.XSHE,瑞和小康,RHXK,2009-11-19,2200-01-01,fja
150009.XSHE,瑞和远见,RHYJ,2009-11-19,2200-01-01,fjb


In [9]:
# 获取所有etf , lof ,fja,fjb等类似
get_all_securities(['etf'])[:2]

,display_name,name,start_date,end_date,type
159001.XSHE,保证金,BZJ,2014-10-20,2200-01-01,etf
159003.XSHE,招商快线,ZSKX,2014-10-20,2200-01-01,etf


### 基金累计/单位净值
得到基金在一段时间的如下的数据:

acc_net_value: 基金累计净值  
unit_net_value: 基金单位净值

**调用方法**

get_extras(info, security_list, start_date='2015-01-01', end_date='2015-12-31', df=True,count=None):

**参数**

info: [‘acc_net_value’, ‘unit_net_value’] 中的一个  
security_list: 基金列表  
start_date/end_date: 开始结束日期, 同[get_price]  
df: 返回[pandas.DataFrame]对象还是一个dict, 同[history]  
count:限定返回条数

**返回值**

df=True:  
[pandas.DataFrame]对象, 列索引是基金代号, 行索引是[datetime.datetime], 比如get_extras('acc_net_value', ['510300.XSHG', '510050.XSHG'], start_date='2015-12-01', end_date='2015-12-03')返回:

In [10]:
get_extras('acc_net_value', ['510300.XSHG', '510050.XSHG'], start_date='2015-12-01', end_date='2015-12-03')

,510300.XSHG,510050.XSHG
2015-12-01,1.3950,3.119
2015-12-02,1.4432,3.251
2015-12-03,1.4535,3.254


df=False  
一个dict, key是基金代号, value是[numpy.ndarray], 比如 get_extras('acc_net_value', ['510300.XSHG', '510050.XSHG'], start_date='2015-12-01', end_date='2015-12-03', df=False) 返回:

In [11]:
get_extras('acc_net_value', ['510300.XSHG', '510050.XSHG'], start_date='2015-12-01', end_date='2015-12-03', df=False)

{'510050.XSHG': array([3.119, 3.251, 3.254]),
 '510300.XSHG': array([1.395 , 1.4432, 1.4535])}

### 场内基金份额数据
from jqdatasdk import *  
finance.run_query(query(finance.FUND_SHARE_DAILY).filter(finance.FUND_SHARE_DAILY.date=='2019-05-23').limit(n))

描述：记录每日场内基金份额数据

**参数：**

query(finance.FUND_SHARE_DAILY)：表示从finance.FUND_SHARE_DAILY这张表中查询每日场内基金份额数据，还可以指定所要查询的字段名，格式如下：query(库名.表名.字段名1，库名.表名.字段名2），多个字段用逗号分隔进行提取；query函数的更多用法详见：query简易教程

finance.FUND_SHARE_DAILY：收录了每日场内基金份额数据，表结构和字段信息如下：

**字段设计：**

 

| 名称          | 类型         | 描述                                                   |
| ------------- | ------------ | ------------------------------------------------------ |
| code          | varchar(12)  | 基金代码                                               |
| name          | varchar(50） | 基金简称                                               |
| exchange_code | varchar(12)  | 交易市场编码，XSHG-上海证券交易所；XSHE-深圳证券交易所 |
| date          | date         | 日期                                                   |
| shares        | bigint       | 基金份额（份）                                         |

filter(finance.FUND_SHARE_DAILY.date==date)：指定筛选条件，通过finance.FUND_SHARE_DAILY.date==date可以指定你想要查询的日期；除此之外，还可以对表中其他字段指定筛选条件；多个筛选条件用英文逗号分隔。

limit(n)：限制返回的数据条数，n指定返回条数。

**返回结果：**

返回一个 dataframe，每一行对应数据表中的一条数据， 列索引是您所查询的字段名称

**注意：**

1. 为了防止返回数据量过大, 我们每次最多返回3000行
2. 不能进行连表查询，即同时查询多张表的数据

**示例：**

In [12]:
#查询2019-05-23的场内基金份额数据。
df=finance.run_query(query(finance.FUND_SHARE_DAILY).filter(finance.FUND_SHARE_DAILY.date=='2019-05-23').limit(10))
df.head()

,id,code,name,exchange_code,date,shares
0,960881,150008.XSHE,瑞和小康,XSHE,2019-05-23,17749200
1,960882,150009.XSHE,瑞和远见,XSHE,2019-05-23,17749200
2,960883,150012.XSHE,中证100A,XSHE,2019-05-23,36139500
3,960884,150013.XSHE,中证100B,XSHE,2019-05-23,54209300
4,960885,150016.XSHE,合润A,XSHE,2019-05-23,38111200


## 基金列表(点击省略号可以展开)
我们目前提供了近600种在交易所上市的基金的行情、净值等数据，包含ETF、LOF、分级AB以及货币基金。

(温馨提示：鉴于内容太多, 可使用Ctrl + F进行搜索)


In [17]:
import pandas as pd
# 设定最大显示行数、列数为10000
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 10000)
pd.set_option('display.width', 10000)


### ETF列表

In [18]:
get_all_securities('etf',date=None)

,display_name,name,start_date,end_date,type
159001.XSHE,保证金,BZJ,2014-10-20,2200-01-01,etf
159003.XSHE,招商快线,ZSKX,2014-10-20,2200-01-01,etf
159005.XSHE,添富快钱,TFKQ,2015-01-13,2200-01-01,etf
159901.XSHE,深100ETF,S100ETF,2006-04-24,2200-01-01,etf
159902.XSHE,中小板,ZXB,2006-09-05,2200-01-01,etf
159903.XSHE,深成ETF,SCETF,2010-02-02,2200-01-01,etf
159905.XSHE,深红利,SHL,2011-01-11,2200-01-01,etf
159906.XSHE,深成长,SCZ,2011-02-23,2200-01-01,etf
159907.XSHE,中小300,ZX300,2011-08-10,2200-01-01,etf
159908.XSHE,博时创业,SF200,2011-07-13,2200-01-01,etf


### LOF列表

In [19]:
get_all_securities('lof',date=None)

,display_name,name,start_date,end_date,type
160105.XSHE,南方积配,NFJP,2004-12-20,2200-01-01,lof
160106.XSHE,南方高增,NFGZ,2005-09-21,2200-01-01,lof
160119.XSHE,南方500,NF500,2009-11-11,2200-01-01,lof
160123.XSHE,南方10年国债A,NF50Z,2011-07-25,2016-08-16,lof
160125.XSHE,南方香港,NFXG,2011-11-01,2200-01-01,lof
160128.XSHE,南方金利,NFJL,2012-07-16,2200-01-01,lof
160130.XSHE,南方永利,NFYL,2013-06-06,2200-01-01,lof
160131.XSHE,南方聚利,NFJL,2014-01-21,2200-01-01,lof
160133.XSHE,南方天元,NFTY,2014-08-25,2200-01-01,lof
160138.XSHE,国企精明,GQJM,2017-05-17,2019-04-22,lof


### 分级基金列表

In [20]:
get_all_securities(['fja','fjb'],date=None)

,display_name,name,start_date,end_date,type
150008.XSHE,瑞和小康,RHXK,2009-11-19,2200-01-01,fja
150009.XSHE,瑞和远见,RHYJ,2009-11-19,2200-01-01,fjb
150012.XSHE,中证100A,ZZ100A,2010-06-18,2200-01-01,fja
150013.XSHE,中证100B,ZZ100B,2010-06-18,2200-01-01,fjb
150016.XSHE,合润A,HRA,2010-05-31,2200-01-01,fja
150017.XSHE,合润B,HRB,2010-05-31,2200-01-01,fjb
150018.XSHE,银华稳进,YHWJ,2010-06-07,2200-01-01,fja
150019.XSHE,银华锐进,YHRJ,2010-06-07,2200-01-01,fjb
150022.XSHE,深成指A,SCZA,2010-11-08,2200-01-01,fja
150023.XSHE,深成指B,SCZB,2010-11-08,2200-01-01,fjb


### 货币基金列表

| 货币基金代码 | 货币基金名称 | 起始日期   | 是否T+0 |
| ------------ | ------------ | ---------- | ------- |
| 159001.XSHE  | 保证金       | 2014-10-20 | 是      |
| 159003.XSHE  | 招商快线     | 2014-10-20 | 是      |
| 159005.XSHE  | 添富快钱     | 2015-01-13 | 是      |
| 511600.XSHG  | 货币ETF      | 2016-09-27 | 是      |
| 511620.XSHG  | 货币基金     | 2017-09-07 | 是      |
| 511650.XSHG  | 华夏快线     | 2017-01-16 | 是      |
| 511660.XSHG  | 建信添益     | 2016-09-21 | 是      |
| 511680.XSHG  | 安信货币     | 2016-09-27 | 是      |
| 511690.XSHG  | 交易货币     | 2016-10-20 | 是      |
| 511700.XSHG  | 场内货币     | 2016-10-17 | 是      |
| 511760.XSHG  | 德邦货币     | 2016-12-15 | 是      |
| 511770.XSHG  | 金鹰增益     | 2017-04-10 | 是      |
| 511800.XSHG  | 易货币       | 2014-12-08 | 是      |
| 511810.XSHG  | 理财金H      | 2015-01-05 | 是      |
| 511820.XSHG  | 鹏华添利     | 2016-02-22 | 是      |
| 511830.XSHG  | 华泰货币     | 2015-08-03 | 是      |
| 511850.XSHG  | 财富宝E      | 2016-07-18 | 是      |
| 511860.XSHG  | 博时货币     | 2014-12-09 | 是      |
| 511880.XSHG  | 银华日利     | 2013-04-18 | 是      |
| 511890.XSHG  | 景顺货币     | 2015-08-03 | 是      |
| 511900.XSHG  | 富国货币     | 2015-12-09 | 是      |
| 511910.XSHG  | 融通货币     | 2016-07-04 | 是      |
| 511920.XSHG  | 广发货币     | 2016-03-28 | 是      |
| 511930.XSHG  | 中融日盈     | 2015-12-15 | 是      |
| 511950.XSHG  | 广发添利     | 2016-12-19 | 是      |
| 511960.XSHG  | 嘉实快线     | 2016-01-11 | 是      |
| 511970.XSHG  | 国寿货币     | 2016-07-18 | 是      |
| 511980.XSHG  | 现金添富     | 2015-11-02 | 是      |
| 511990.XSHG  | 华宝添益     | 2013-01-28 | 是      |